In [13]:
import pandas as pd
import json
import matplotlib.pyplot as plt

In [18]:
experiments = [
    # {"id": "exp_20250626_205804", "name": "Openwhisk"},
    {"id": "exp_20250625_113106", "name": "NMIG"},
    # {"id": "exp_20250626_155721", "name": "Histogram"},
    # {"id": "exp_20250626_113035", "name": "Pagurus"},
]

mem_cost_per_sec_per_mb = 0.00001
cpu_cost_per_sec= 0.00005
memory_mb = 1024

In [19]:


plot_data = {}
summary = {}

for exp in experiments:
    path = f"../results/{exp['id']}/docker_log.csv"
    df = pd.read_csv(path, names=['Timestamp', 'name', 'image', 'status'])

    timestamps = sorted(df['Timestamp'].unique())

    # container status per timestamp (same as your code)
    container_status = {}
    for name, group in df.groupby('name'):
        group = group.sort_values('Timestamp')
        status_list = []
        last_status = None
        idx = 0
        for t in timestamps:
            while idx < len(group) and group.iloc[idx]['Timestamp'] <= t:
                last_status = group.iloc[idx]['status']
                idx += 1
            status_list.append(last_status)
        container_status[name] = status_list

    cum_mem_mbs = []      # cumulative memory-time, MB·s
    cum_cpu_s = []        # cumulative CPU-time, s
    total_mem_mbs = 0.0
    total_cpu_s = 0.0

    for i in range(1, len(timestamps)):
        interval_sec = (timestamps[i] - timestamps[i-1]) / 1000.0
        for name, status_list in container_status.items():
            status = status_list[i-1]
            if status is None:
                continue
            # memory is held whether running or paused
            if status in ('running', 'paused'):
                total_mem_mbs += interval_sec * memory_mb
            # CPU consumed only while running
            if status == 'running':
                total_cpu_s += interval_sec
        cum_mem_mbs.append(total_mem_mbs)
        cum_cpu_s.append(total_cpu_s)

    elapsed_secs = [(t - timestamps[0]) / 1000 for t in timestamps[1:]]
    plot_data[exp['name']] = (elapsed_secs, cum_mem_mbs, cum_cpu_s)
    summary[exp['name']] = {
        "memory_MB_s": total_mem_mbs,
        "memory_GB_s": total_mem_mbs / 1024.0,
        "cpu_s": total_cpu_s,
    }

# print the summary table
print(f"{'Method':12s} {'Mem (GB·s)':>14s} {'CPU (s)':>12s}")
for name, s in summary.items():
    print(f"{name:12s} {s['memory_GB_s']:>14.1f} {s['cpu_s']:>12.1f}")

Method           Mem (GB·s)      CPU (s)
NMIG               326940.0      89890.0


In [20]:
plot_data

{'NMIG': ([1.0,
   2.0,
   3.0,
   5.0,
   6.0,
   7.0,
   8.0,
   9.0,
   10.0,
   11.0,
   12.0,
   13.0,
   14.0,
   15.0,
   16.0,
   17.0,
   18.0,
   19.0,
   20.0,
   21.0,
   22.0,
   23.0,
   24.0,
   25.0,
   26.0,
   27.0,
   28.0,
   29.0,
   30.0,
   31.0,
   32.0,
   33.0,
   34.0,
   35.0,
   36.0,
   38.0,
   39.0,
   40.0,
   41.0,
   42.0,
   43.0,
   44.0,
   45.0,
   46.0,
   47.0,
   48.0,
   49.0,
   50.0,
   51.0,
   52.0,
   53.0,
   54.0,
   55.0,
   56.0,
   57.0,
   58.0,
   59.0,
   60.0,
   61.0,
   62.0,
   63.0,
   64.0,
   65.0,
   66.0,
   67.0,
   68.0,
   69.0,
   71.0,
   72.0,
   73.0,
   74.0,
   75.0,
   76.0,
   77.0,
   78.0,
   79.0,
   80.0,
   81.0,
   82.0,
   83.0,
   84.0,
   85.0,
   86.0,
   87.0,
   88.0,
   89.0,
   90.0,
   91.0,
   92.0,
   93.0,
   94.0,
   95.0,
   96.0,
   97.0,
   98.0,
   99.0,
   100.0,
   102.0,
   103.0,
   104.0,
   105.0,
   106.0,
   107.0,
   108.0,
   109.0,
   110.0,
   111.0,
   112.0,
   113.0,
   114